In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import random
import re
import os
import glob
from datetime import date


In [2]:
ARCHIVE_DIR = "archive"

files = glob.glob(os.path.join(ARCHIVE_DIR, "*_weekdays.csv")) + glob.glob(os.path.join(ARCHIVE_DIR, "*_weekends.csv"))

dfs = []
for f in files:
    base = os.path.basename(f).replace(".csv", "")
    city, week_type = base.rsplit("_", 1)
    d = pd.read_csv(f)
    d["city"] = city.capitalize()
    d["week_type"] = week_type
    dfs.append(d)

master_df = pd.concat(dfs, ignore_index=True)
print(master_df.shape)
master_df.head()


(51707, 22)


,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,...,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,week_type
0,0,194.033698,Private room,False,True,2.0,False,1,0,10.0,...,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,Amsterdam,weekdays
1,1,344.245776,Private room,False,True,4.0,False,0,0,8.0,...,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,Amsterdam,weekdays
2,2,264.101422,Private room,False,True,2.0,False,0,1,9.0,...,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,Amsterdam,weekdays
3,3,433.529398,Private room,False,True,4.0,False,0,1,9.0,...,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663,Amsterdam,weekdays
4,4,485.552926,Private room,False,True,2.0,True,0,0,10.0,...,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508,Amsterdam,weekdays


In [3]:
filtered_df = master_df[(master_df["room_type"] == "Entire home/apt") & (master_df["bedrooms"] == 2)]
print(filtered_df.shape)
filtered_df.groupby("city").size()


(8650, 22)


city
Amsterdam     354
Athens       1489
Barcelona     239
Berlin        174
Budapest      815
Lisbon       1241
London       1307
Paris         719
Rome         1756
Vienna        556
dtype: int64

In [4]:
CITY_COUNTRY = {
    "Amsterdam": "Netherlands",
    "Athens": "Greece",
    "Barcelona": "Spain",
    "Berlin": "Germany",
    "Budapest": "Hungary",
    "Lisbon": "Portugal",
    "London": "United Kingdom",
    "Paris": "France",
    "Rome": "Italy",
    "Vienna": "Austria",
}

CITIES = [(c, CITY_COUNTRY[c]) for c in sorted(filtered_df["city"].unique().tolist())]
print(CITIES)


[('Amsterdam', 'Netherlands'), ('Athens', 'Greece'), ('Barcelona', 'Spain'), ('Berlin', 'Germany'), ('Budapest', 'Hungary'), ('Lisbon', 'Portugal'), ('London', 'United Kingdom'), ('Paris', 'France'), ('Rome', 'Italy'), ('Vienna', 'Austria')]


In [5]:
TARGET_PER_CITY = 40
PAGE_LOAD_WAIT = 8

OUTPUT_DIR = "bronze"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "airbnb_raw_data.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [6]:
def get_rating_reviews_fav(driver, wait):
    rating, reviews, is_fav = None, None, 0

    try:
        try:
            review_link = wait.until(EC.presence_of_element_located(
                (By.XPATH, "//a[contains(@href, 'reviews')] | //button[contains(., 'reviews')]")
            ))
        except Exception:
            try:
                h1_parent = driver.find_element(By.TAG_NAME, "h1").find_element(By.XPATH, "./..").text
                if "New" in h1_parent:
                    return "New", 0, 0
            except Exception:
                pass
            return rating, reviews, is_fav

        link_text = review_link.get_attribute("textContent").strip()
        clean_link = " ".join(link_text.split())

        parent_text = review_link.find_element(By.XPATH, "./..").get_attribute("textContent").strip()
        clean_parent = " ".join(parent_text.split())

        if "Guest favorite" in clean_link or "Guest favorite" in clean_parent:
            is_fav = 1
            r_match = re.search(r"(\d\.\d{1,2})", clean_parent)
            if r_match:
                rating = r_match.group(1)
            rev_match = re.search(r"(\d+)\s*Review", clean_parent, re.IGNORECASE)
            if rev_match:
                reviews = rev_match.group(1)
        else:
            is_fav = 0
            if "New" in clean_parent:
                rating = "New"
            else:
                r_match = re.search(r"(\d\.\d{1,2})", clean_parent)
                if r_match:
                    rating = r_match.group(1)

            rev_match = re.search(r"(\d+)\s*review", clean_link, re.IGNORECASE)
            if not rev_match:
                rev_match = re.search(r"(\d+)\s*review", clean_parent, re.IGNORECASE)
            if rev_match:
                reviews = rev_match.group(1)

    except Exception:
        pass

    return rating, reviews, is_fav


In [7]:
def get_property_details(body_text):
    property_type, guests, bedrooms, beds, baths, host_name = (None,) * 6

    type_match = re.search(r"(Entire [A-Za-z/ ]+|Private room[A-Za-z/ ]*|Shared room[A-Za-z/ ]*|Hotel room[A-Za-z/ ]*)", body_text)
    if type_match:
        property_type = type_match.group(1).split(" in ")[0].strip()

    guests_match = re.search(r"(\d+)\s*guests?", body_text, re.IGNORECASE)
    if guests_match:
        guests = int(guests_match.group(1))

    bedrooms_match = re.search(r"(\d+)\s*bedrooms?", body_text, re.IGNORECASE)
    if bedrooms_match:
        bedrooms = int(bedrooms_match.group(1))
    elif re.search(r"studio", body_text, re.IGNORECASE):
        bedrooms = 0

    beds_match = re.search(r"(\d+)\s*beds?\b", body_text, re.IGNORECASE)
    if beds_match:
        beds = int(beds_match.group(1))

    baths_match = re.search(r"(\d+(?:\.\d+)?)\s*(?:shared\s*)?baths?", body_text, re.IGNORECASE)
    if baths_match:
        baths = float(baths_match.group(1))

    host_match = re.search(r"Hosted by ([A-Za-z ]{2,30})", body_text)
    if host_match:
        host_name = host_match.group(1).strip()

    return property_type, guests, bedrooms, beds, baths, host_name


CURRENCY_CODES = r"(?:NOK|EUR|USD|GBP|CHF|CZK|HUF|SEK|DKK|PLN)"
NUMBER = r"\d[\d.,\s]{0,10}\d|\d"


def get_price(body_text):
    clean = " ".join(body_text.split())

    patterns = [
        rf"[\$\u00a3\u20ac]\s?(?:{NUMBER})",
        rf"(?:{NUMBER})\s?kr\s?{CURRENCY_CODES}?\b",
        rf"(?:{NUMBER})\s?{CURRENCY_CODES}\b",
    ]

    for p in patterns:
        m = re.search(p, clean)
        if m:
            return m.group(0).strip()

    return None


In [8]:
all_data = []
city_count = 0
scrape_date = date.today().isoformat()

for city, country in CITIES:
    city_count += 1

    URL = (
        f"https://www.airbnb.com/s/{city}/homes?refinement_paths%5B%5D=%2Fhomes"
        f"&date_picker_type=calendar&search_type=filter_change&flexible_trip_lengths%5B%5D=one_week"
        f"&monthly_start_date=2026-02-01&monthly_length=3&monthly_end_date=2026-05-01"
        f"&price_filter_input_type=2&channel=EXPLORE&price_filter_num_nights=5"
        f"&min_bedrooms=2&selected_filter_order%5B%5D=min_bedrooms%3A2"
        f"&selected_filter_order%5B%5D=room_types%3AEntire%20home%2Fapt"
        f"&update_selected_filters=true&query={city}&search_mode=regular_search"
        f"&room_types%5B%5D=Entire%20home%2Fapt"
    )

    chrome_options = Options()
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--start-maximized")
    service = ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    items_scraped_this_city = 0

    try:
        driver.get(URL)
        wait = WebDriverWait(driver, PAGE_LOAD_WAIT)

        while items_scraped_this_city < TARGET_PER_CITY:
            print(f"=== SCRAPING CITY {city_count}/{len(CITIES)}: {city}, {country} ===")
            print(f"--- Collected so far: {len(all_data)} ---")

            try:
                wait.until(EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[data-testid='card-container']")
                ))
            except Exception:
                break

            listings = driver.find_elements(By.CSS_SELECTOR, "div[data-testid='card-container']")
            listing_urls = []
            for l in listings:
                try:
                    url = l.find_element(By.TAG_NAME, "a").get_attribute("href")
                    listing_urls.append(url)
                except Exception:
                    continue

            main_window = driver.current_window_handle

            for url in listing_urls:
                if items_scraped_this_city >= TARGET_PER_CITY:
                    break

                try:
                    driver.execute_script(f"window.open('{url}', '_blank');")
                    driver.switch_to.window(driver.window_handles[-1])
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

                    try:
                        name = driver.find_element(By.TAG_NAME, "h1").text
                    except Exception:
                        name = None

                    driver.execute_script("window.scrollBy(0, 500);")
                    time.sleep(1.5)

                    try:
                        body_text = driver.find_element(By.TAG_NAME, "body").get_attribute("textContent")
                    except Exception:
                        body_text = ""

                    is_superhost = 1 if "Superhost" in body_text else 0
                    price = get_price(body_text)
                    rating, reviews, is_fav = get_rating_reviews_fav(driver, wait)
                    property_type, guests, bedrooms, beds, baths, host_name = get_property_details(body_text)

                    all_data.append({
                        "listing_name": name,
                        "city": city,
                        "country": country,
                        "price_raw": price,
                        "property_type": property_type,
                        "guests": guests,
                        "bedrooms": bedrooms,
                        "beds": beds,
                        "baths": baths,
                        "host_name": host_name,
                        "is_superhost": is_superhost,
                        "is_guest_favorite": is_fav,
                        "rating": rating,
                        "reviews": reviews,
                        "scrape_date": scrape_date,
                        "url": url,
                    })

                    print(f"[{len(all_data)}] {price} | {city} | Rating:{rating} ({reviews}) | Fav:{is_fav} | SH:{is_superhost}")
                    time.sleep(random.uniform(1.0, 2.0))

                    driver.close()
                    driver.switch_to.window(main_window)
                    items_scraped_this_city += 1

                except Exception:
                    if len(driver.window_handles) > 1:
                        driver.close()
                        driver.switch_to.window(main_window)
                    continue

            if items_scraped_this_city < TARGET_PER_CITY:
                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, "a[aria-label='Next']")
                    driver.execute_script("arguments[0].click();", next_btn)
                    time.sleep(4)
                except Exception:
                    break

    finally:
        print(f"Done with {city}. Total collected so far: {len(all_data)}")
        driver.quit()


=== SCRAPING CITY 1/10: Amsterdam, Netherlands ===
--- Collected so far: 0 ---
[1] None | Amsterdam | Rating:None (None) | Fav:0 | SH:1
=== SCRAPING CITY 1/10: Amsterdam, Netherlands ===
--- Collected so far: 1 ---
[2] None | Amsterdam | Rating:4.8 (5) | Fav:0 | SH:1
[3] None | Amsterdam | Rating:None (None) | Fav:0 | SH:1
[4] None | Amsterdam | Rating:None (None) | Fav:0 | SH:0
[5] None | Amsterdam | Rating:4.96 (96139) | Fav:1 | SH:1
[6] € 300 | Amsterdam | Rating:4.77 (770) | Fav:0 | SH:1
[7] None | Amsterdam | Rating:5.0 (2) | Fav:0 | SH:1
[8] None | Amsterdam | Rating:4.99 (9997) | Fav:1 | SH:1
[9] None | Amsterdam | Rating:None (None) | Fav:0 | SH:1
[10] None | Amsterdam | Rating:None (None) | Fav:0 | SH:1
[11] None | Amsterdam | Rating:4.62 (29) | Fav:0 | SH:1
[12] None | Amsterdam | Rating:None (None) | Fav:0 | SH:1
[13] None | Amsterdam | Rating:4.56 (205) | Fav:0 | SH:1
[14] €25.00 | Amsterdam | Rating:None (None) | Fav:0 | SH:1
[15] None | Amsterdam | Rating:4.94 (115) | Fav

In [9]:
df_scraped = pd.DataFrame(all_data)
print(df_scraped.shape)
df_scraped.head()


(353, 16)


,listing_name,city,country,price_raw,property_type,guests,bedrooms,beds,baths,host_name,is_superhost,is_guest_favorite,rating,reviews,scrape_date,url
0,Lovely Apartment with Parking,Amsterdam,Netherlands,None,Entire rental unit,8.0,3.0,3.0,3.0,EricNew HostEnjoy your stay in,1,0,None,None,2026-07-07,https://www.airbnb.com/rooms/17227175490672516...
1,Light apartment next to Vondelpark,Amsterdam,Netherlands,None,Entire rental unit,2.0,2.0,1.0,1.0,Kimo En Marieke,1,0,4.8,5,2026-07-07,https://www.airbnb.com/rooms/14711794?search_m...
2,Modern Apartment Parking,Amsterdam,Netherlands,None,Entire rental unit,7.0,3.0,3.0,3.0,StefanNew HostListing highligh,1,0,None,None,2026-07-07,https://www.airbnb.com/rooms/17218898431659363...
3,Stay tuned,Amsterdam,Netherlands,None,None,NaN,NaN,NaN,NaN,None,0,0,None,None,2026-07-07,https://www.airbnb.com/rooms/70363368112394565...
4,2 Floor Apartment with Large kitchen Nieuw-Vennep,Amsterdam,Netherlands,None,Entire rental unit,5.0,2.0,3.0,1.0,Chris,1,1,4.96,96139,2026-07-07,https://www.airbnb.com/rooms/92675576315069414...


In [10]:
df_scraped.to_csv(OUTPUT_FILE, index=False)
print(OUTPUT_FILE)


bronze\airbnb_raw_data.csv


In [11]:
print(df_scraped.isna().sum())
print(df_scraped["city"].value_counts())


listing_name           0
city                   0
country                0
price_raw            261
property_type          9
guests                 9
bedrooms               9
beds                  15
baths                 10
host_name             15
is_superhost           0
is_guest_favorite      0
rating                47
reviews               47
scrape_date            0
url                    0
dtype: int64
city
Amsterdam    40
Athens       40
Barcelona    40
Budapest     40
Lisbon       40
London       40
Rome         40
Vienna       40
Berlin       33
Name: count, dtype: int64
